In [1]:
import os, glob, time
import numpy as np, pandas as pd, torch, soundfile as sf
from scipy.stats import spearmanr
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, AutoTokenizer, AutoModelForCausalLM

SMOKE = False       # True: 40 train files to check everything. False: full run for the commit.
WORK = '/kaggle/working'

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
folds = pd.read_csv(find('folds.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
uids = list(folds.uid) + ['test/' + f for f in test.filename]
run_uids = uids[:40] if SMOKE else uids
print('files to process:', len(run_uids))

# ---------- 0. prompt and span logic, checked on edge cases before any heavy work ----------
llm_name = 'Qwen/Qwen2.5-7B-Instruct'
tok = AutoTokenizer.from_pretrained(llm_name)
SYS = 'You are an expert examiner of spoken English.'
USER = ('Below is an automatic word-by-word transcript of a candidate speaking for about a minute. '
        "It has no punctuation and keeps the speaker's own words, including grammatical mistakes. "
        'Some words are misrecognized and spelled the way they sounded; ignore spelling and judge only grammar.\n\n'
        'Transcript: {t}\n\n'
        'Rate the grammatical accuracy of the speech from 1 (many serious errors) to 5 (almost no errors). '
        'Answer with a single digit.')
MARK = 'Transcript: '

def build(t):
    t = t if t else '(no speech)'
    s = tok.apply_chat_template([{'role': 'system', 'content': SYS},
                                 {'role': 'user', 'content': USER.format(t=t)}],
                                tokenize=False, add_generation_prompt=True)
    a = s.find(MARK + t)
    assert a >= 0
    a += len(MARK)
    b = a + len(t)
    enc = tok(s, return_offsets_mapping=True, return_tensors='pt', add_special_tokens=False)
    offs = enc.pop('offset_mapping')[0]
    mask = (offs[:, 1] > a) & (offs[:, 0] < b) & (offs[:, 1] > offs[:, 0])   # any token touching the span
    return enc, mask

for t in ['a', 'i', 'the', '', 'a a a', 'my favorite place to visit is the beach']:
    _, mk = build(t)
    assert mk.sum() > 0, repr(t)
print('span check passed on edge cases')

# ---------- 1. literal CTC transcripts (no language model) ----------
asr_name = 'facebook/wav2vec2-large-960h-lv60-self'
proc = Wav2Vec2Processor.from_pretrained(asr_name)
asr = Wav2Vec2ForCTC.from_pretrained(asr_name).to('cuda:0').half().eval()
texts, t0 = [], time.time()
for i, u in enumerate(run_uids):
    wav, sr = sf.read(f'{BASE}/{u}', dtype='float32')
    assert sr == 16000, (u, sr)
    if wav.ndim > 1:
        wav = wav.mean(1)
    x = proc(wav, sampling_rate=16000, return_tensors='pt').input_values.to('cuda:0').half()
    with torch.inference_mode():
        ids = asr(x).logits.argmax(-1)
    texts.append(proc.batch_decode(ids)[0].lower().strip())
    if (i + 1) % 200 == 0:
        print(f'ctc {i+1}/{len(run_uids)} | {(time.time()-t0)/60:.1f} min')
del asr
torch.cuda.empty_cache()

T = pd.DataFrame({'uid': run_uids, 'text': texts})
T['n_words'] = T.text.str.split().str.len()
T.to_csv(f'{WORK}/ctc_transcripts.csv', index=False)          # saved before the LLM step
TF = pd.read_csv(find('text_features.csv'), index_col='uid')
lab = folds.set_index('uid')
is_tr = T.uid.isin(lab.index).values
print('word count corr, CTC vs Whisper:', round(np.corrcoef(T.n_words, TF.loc[T.uid, 'n_words'])[0, 1], 3))
print('empty transcripts:', int((T.n_words == 0).sum()), '| of which train zero-label files:',
      int(((T.n_words == 0).values & is_tr & (lab.reindex(T.uid).is_zero.values == 1)).sum()))
print('shortest non-empty transcripts:', T[T.n_words > 0].sort_values('n_words').head(5)[['uid', 'n_words', 'text']].values.tolist())

# ---------- 2. Qwen2.5-7B mean-pooled hidden states over the CTC transcript ----------
llm = AutoModelForCausalLM.from_pretrained(llm_name, dtype=torch.float16, device_map='auto').eval()
POOL = [6, 9, 12, 16, 20, 24]                       # same layers as notebook 11
digit_ids = [tok.encode(str(k), add_special_tokens=False)[0] for k in range(1, 6)]
assert len(set(digit_ids)) == 5

pooled = np.zeros((len(run_uids), len(POOL), llm.config.hidden_size), dtype=np.float16)
rows, t0 = [], time.time()
for i, (u, t) in enumerate(zip(T.uid, T.text)):
    enc, mask = build(t)
    assert mask.sum() > 0, u
    with torch.inference_mode():
        out = llm(**{k: v.to('cuda:0') for k, v in enc.items()}, output_hidden_states=True)
    for j, L in enumerate(POOL):
        h = out.hidden_states[L][0]
        pooled[i, j] = h[mask.to(h.device)].float().mean(0).cpu().numpy()
    p = out.logits[0, -1, digit_ids].float().softmax(-1).cpu().numpy()
    rows.append(dict(uid=u, e=float((p * np.arange(1, 6)).sum()), ent=float(-(p * np.log(p + 1e-12)).sum()),
                     **{f'p{k+1}': float(p[k]) for k in range(5)}, n_tok=int(mask.sum())))
    if (i + 1) % 100 == 0:
        print(f'llm {i+1}/{len(run_uids)} | {(time.time()-t0)/60:.1f} min')

S = pd.DataFrame(rows)
print('NaN or inf in pooled:', bool(~np.isfinite(pooled.astype(np.float32)).all()),
      '| largest abs:', float(np.abs(pooled.astype(np.float32)).max()))
print('transcript tokens per file, min/median/max:', S.n_tok.min(), int(S.n_tok.median()), S.n_tok.max())
tr = S[S.uid.isin(lab.index)].copy()
tr['label'] = lab.loc[tr.uid, 'label'].values
tr = tr[lab.loc[tr.uid, 'is_zero'].values == 0]
for g in [None, 'dur_45s', 'dur_60s']:
    sub = tr if g is None else tr[lab.loc[tr.uid, 'dur_group'].values == g]
    print(f'Spearman(zero-shot score on CTC transcript, label) {g or "all"} n={len(sub)}:',
          round(spearmanr(sub.e, sub.label).correlation, 3))
print('reference, 7B on Whisper transcript: all 0.622 | 45s 0.657 | 60s 0.627')

# ---------- save ----------
if not SMOKE:
    assert len(T) == 985 and pooled.shape == (985, 6, 3584)
S.to_csv(f'{WORK}/llmc_scores.csv', index=False)
pd.DataFrame({'uid': run_uids}).to_csv(f'{WORK}/llmc_uids.csv', index=False)
np.save(f'{WORK}/llmc_meanpool.npy', pooled)
print('saved', os.listdir(WORK))

files to process: 985


span check passed on edge cases


[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ctc 200/985 | 1.4 min
ctc 400/985 | 2.9 min
ctc 600/985 | 4.4 min
ctc 800/985 | 5.9 min
word count corr, CTC vs Whisper: 0.781
empty transcripts: 30 | of which train zero-label files: 30
[per-file rows removed before publishing]


llm 100/985 | 0.3 min
llm 200/985 | 0.7 min
llm 300/985 | 1.0 min
llm 400/985 | 1.3 min
llm 500/985 | 1.6 min
llm 600/985 | 2.0 min
llm 700/985 | 2.3 min
llm 800/985 | 2.7 min
llm 900/985 | 3.0 min
NaN or inf in pooled: False | largest abs: 126.75
transcript tokens per file, min/median/max: 1 110 185
Spearman(zero-shot score on CTC transcript, label) all n=732: 0.533
Spearman(zero-shot score on CTC transcript, label) dur_45s n=154: 0.429
Spearman(zero-shot score on CTC transcript, label) dur_60s n=549: 0.618
reference, 7B on Whisper transcript: all 0.622 | 45s 0.657 | 60s 0.627
saved ['llmc_meanpool.npy', 'llmc_scores.csv', 'ctc_transcripts.csv', 'llmc_uids.csv', '__notebook__.ipynb']
